In [1]:
import h5torch
import os
cell_line_selection = ['GM12878', 'K562', 'HepG2', 'A549']
# Get all h5t files from the specified folder
h5t_files = [os.path.join('/data/home/natant/Negatives/Data/Encode690/ENCODE_hg38_subset_101bp_celltypes_ATAC_H5_all_chr', f) 
             for f in os.listdir('/data/home/natant/Negatives/Data/Encode690/ENCODE_hg38_subset_101bp_celltypes_ATAC_H5_all_chr') 
             if f.endswith('.h5t') and f.split('.')[0] in cell_line_selection]

In [2]:
h5t_files

['/data/home/natant/Negatives/Data/Encode690/ENCODE_hg38_subset_101bp_celltypes_ATAC_H5_all_chr/GM12878.h5t',
 '/data/home/natant/Negatives/Data/Encode690/ENCODE_hg38_subset_101bp_celltypes_ATAC_H5_all_chr/HepG2.h5t',
 '/data/home/natant/Negatives/Data/Encode690/ENCODE_hg38_subset_101bp_celltypes_ATAC_H5_all_chr/K562.h5t',
 '/data/home/natant/Negatives/Data/Encode690/ENCODE_hg38_subset_101bp_celltypes_ATAC_H5_all_chr/A549.h5t']

In [3]:
TFs_per_cell_line = {}
for h5t_file in h5t_files:
    with h5torch.File(h5t_file, 'r') as h5f:
        #print(h5f["0/prot_names"][:].astype(str))
        cell_line = os.path.basename(h5t_file).split('.')[0]
        TFs_per_cell_line[cell_line] = [tf for tf in h5f["0/prot_names"][:].astype(str).tolist() if tf != 'ATAC_peak']

In [4]:
# Convert each TF list to a set for easier comparison
tf_sets = {cell_line: set(tfs) for cell_line, tfs in TFs_per_cell_line.items()}

# Find TFs present in all cell lines (complete overlap)
common_tfs_all = set.intersection(*tf_sets.values())
print(f"TFs common to all cell lines ({len(common_tfs_all)}): {sorted(common_tfs_all)}")

# Find pairwise overlaps
print("\nPairwise overlaps:")
cell_lines = list(TFs_per_cell_line.keys())
for i in range(len(cell_lines)):
    for j in range(i+1, len(cell_lines)):
        overlap = tf_sets[cell_lines[i]] & tf_sets[cell_lines[j]]
        print(f"{cell_lines[i]} ∩ {cell_lines[j]}: {len(overlap)} TFs")

# Find TFs unique to each cell line
print("\nUnique TFs per cell line:")
for cell_line in cell_lines:
    unique_tfs = tf_sets[cell_line] - set.union(*[tf_sets[cl] for cl in cell_lines if cl != cell_line])
    print(f"{cell_line}: {len(unique_tfs)} unique TFs - {sorted(unique_tfs) if unique_tfs else 'None'}")

TFs common to all cell lines (7): ['ATF3', 'CTCF', 'ELF1_(SC-631)', 'Max', 'USF-1', 'YY1_(SC-281)', 'ZBTB33']

Pairwise overlaps:
GM12878 ∩ HepG2: 21 TFs
GM12878 ∩ K562: 27 TFs
GM12878 ∩ A549: 10 TFs
HepG2 ∩ K562: 22 TFs
HepG2 ∩ A549: 11 TFs
K562 ∩ A549: 9 TFs

Unique TFs per cell line:
GM12878: 6 unique TFs - ['ATF2_(SC-81188)', 'FOXM1_(SC-502)', 'IKZF1_(IkN)_(UCLA)', 'Pbx3', 'ZEB1_(SC-25388)', 'ZZZ3']
HepG2: 6 unique TFs - ['ARID3A_(NB100-279)', 'CEBPD_(SC-636)', 'HSF1', 'IRF3', 'MYBL2_(SC-81192)', 'TCF7L2']
K562: 5 unique TFs - ['ATF1_(06-325)', 'Bach1_(sc-14700)', 'FOSL1_(SC-183)', 'NR2F2_(SC-271940)', 'SETDB1']
A549: 1 unique TFs - ['CREB1_(SC-240)']


In [5]:
GM12878_overlaps = {}
# Find pairwise overlaps
print("\nPairwise overlaps:")
cell_lines = list(TFs_per_cell_line.keys())
for cell_line in ['K562', 'HepG2', 'A549']:
    overlap = tf_sets['GM12878'] & tf_sets[cell_line]
    GM12878_overlaps[cell_line] = overlap
    print(f"GM12878 ∩ {cell_line}: {len(overlap)} TFs")




Pairwise overlaps:
GM12878 ∩ K562: 27 TFs
GM12878 ∩ HepG2: 21 TFs
GM12878 ∩ A549: 10 TFs


In [6]:
import os
import pandas as pd
import re
convert_dict = {
    "dinucl_sampled": "dinucl-sampled",
    "dinucl_shuffled": "dinucl-shuffled",
}
# Define the folder path
folder_path = "/data/home/natant/Negatives/Runs/Review_rerun"

# Get all files ending in .ckpt
ckpt_files = [f for f in os.listdir(folder_path) if f.endswith('.ckpt')]

# Parse each filename to extract information
data = []
negative_types = ["dinucl-sampled", "celltype", "shuffled", "dinucl-shuffled", "neighbors"]
for filename in ckpt_files:   
    old_filename = filename
    for old_name, new_name in convert_dict.items():
        filename = filename.replace(old_name, new_name)
    parts = filename.split('_')
    neg_type = parts[-8]
    if neg_type not in negative_types:
        neg_type = "HQ"
        tf = '_'.join(parts[1:-7])
    else:
        tf = '_'.join(parts[1:-8])
        neg_type = parts[-8]

    
    # Extract cell line (first part)
    cellline = parts[0] 
    cv_split = parts[-7].split('-')[1]


    
    
    data.append({
        'filename': filename,
        'file_path': os.path.join(folder_path, old_filename),
        'cellline': cellline,
        'TF': tf,
        'negative_type': neg_type,
        'cv_split': cv_split
    })

# Create DataFrame
df_ckpt = pd.DataFrame(data)
df_ckpt

,filename,file_path,cellline,TF,negative_type,cv_split
0,GM12878_TCF12_dinucl-sampled_CV-4_20251031_02:...,/data/home/natant/Negatives/Runs/Review_rerun/...,GM12878,TCF12,dinucl-sampled,4
1,GM12878_NFIC_(SC-81335)_celltype_CV-4_20251103...,/data/home/natant/Negatives/Runs/Review_rerun/...,GM12878,NFIC_(SC-81335),celltype,4
2,GM12878_ETS1_celltype_CV-0_20251103_21:31_epoc...,/data/home/natant/Negatives/Runs/Review_rerun/...,GM12878,ETS1,celltype,0
3,GM12878_SIX5_shuffled_CV-2_20251030_19:51_epoc...,/data/home/natant/Negatives/Runs/Review_rerun/...,GM12878,SIX5,shuffled,2
4,GM12878_ZBTB33_celltype_CV-0_20251103_21:12_ep...,/data/home/natant/Negatives/Runs/Review_rerun/...,GM12878,ZBTB33,celltype,0
...,...,...,...,...,...,...
1423,GM12878_TBP_shuffled_CV-4_20251030_16:28_epoch...,/data/home/natant/Negatives/Runs/Review_rerun/...,GM12878,TBP,shuffled,4
1424,GM12878_NFIC_(SC-81335)_celltype_CV-1_20251103...,/data/home/natant/Negatives/Runs/Review_rerun/...,GM12878,NFIC_(SC-81335),celltype,1
1425,HepG2_FOXA1_(SC-101058)_neighbors_CV-5_2025103...,/data/home/natant/Negatives/Runs/Review_rerun/...,HepG2,FOXA1_(SC-101058),neighbors,5
1426,GM12878_ATF2_(SC-81188)_shuffled_CV-3_20251030...,/data/home/natant/Negatives/Runs/Review_rerun/...,GM12878,ATF2_(SC-81188),shuffled,3


In [7]:
total_runs = 0
for cell_line in GM12878_overlaps:
    for tf in GM12878_overlaps[cell_line]:
        selected_runs = df_ckpt[(df_ckpt['cellline'] == 'GM12878') & (df_ckpt['TF'] == tf)]
        print(f"TF: {tf}, Runs found: {len(selected_runs)}")
        total_runs += len(selected_runs)
print(f"Total runs found: {total_runs}")

TF: USF2, Runs found: 36
TF: SRF, Runs found: 36
TF: Nrf1, Runs found: 36
TF: ZBTB33, Runs found: 36
TF: RFX5_(200-401-194), Runs found: 36
TF: JunD, Runs found: 36
TF: Mxi1_(AF4185), Runs found: 36
TF: SIX5, Runs found: 36
TF: STAT5A_(SC-74442), Runs found: 36
TF: ETS1, Runs found: 36
TF: SP1, Runs found: 36
TF: CTCF, Runs found: 36
TF: ELK1_(1277-1), Runs found: 36
TF: ATF3, Runs found: 36
TF: TBP, Runs found: 36
TF: USF-1, Runs found: 36
TF: CEBPB_(SC-150), Runs found: 36
TF: ELF1_(SC-631), Runs found: 36
TF: YY1_(SC-281), Runs found: 36
TF: Egr-1, Runs found: 36
TF: MEF2A, Runs found: 36
TF: MAZ_(ab85725), Runs found: 36
TF: Max, Runs found: 36
TF: Znf143_(16618-1-AP), Runs found: 36
TF: NF-YB, Runs found: 36
TF: NF-YA, Runs found: 36
TF: ZNF274, Runs found: 36
TF: USF2, Runs found: 36
TF: SRF, Runs found: 36
TF: Nrf1, Runs found: 36
TF: ZBTB33, Runs found: 36
TF: RXRA, Runs found: 36
TF: RFX5_(200-401-194), Runs found: 36
TF: JunD, Runs found: 36
TF: Mxi1_(AF4185), Runs found: 36


In [8]:
all_runs = []
for cell_line in GM12878_overlaps:
    for tf in GM12878_overlaps[cell_line]:
        selected_runs = df_ckpt[(df_ckpt['cellline'] == 'GM12878') & (df_ckpt['TF'] == tf) & (df_ckpt['negative_type'] != 'HQ')]
        selected_runs["cellline_test"] = cell_line
        all_runs.append(selected_runs)
all_runs_df = pd.concat(all_runs, ignore_index=True)
all_runs_df

/tmp/ipykernel_1168808/4105315406.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  selected_runs["cellline_test"] = cell_line
/tmp/ipykernel_1168808/4105315406.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  selected_runs["cellline_test"] = cell_line
/tmp/ipykernel_1168808/4105315406.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/p

,filename,file_path,cellline,TF,negative_type,cv_split,cellline_test
0,GM12878_USF2_neighbors_CV-1_20251031_00:12_epo...,/data/home/natant/Negatives/Runs/Review_rerun/...,GM12878,USF2,neighbors,1,K562
1,GM12878_USF2_shuffled_CV-1_20251031_00:03_epoc...,/data/home/natant/Negatives/Runs/Review_rerun/...,GM12878,USF2,shuffled,1,K562
2,GM12878_USF2_dinucl-sampled_CV-0_20251030_23:4...,/data/home/natant/Negatives/Runs/Review_rerun/...,GM12878,USF2,dinucl-sampled,0,K562
3,GM12878_USF2_dinucl-sampled_CV-5_20251030_23:5...,/data/home/natant/Negatives/Runs/Review_rerun/...,GM12878,USF2,dinucl-sampled,5,K562
4,GM12878_USF2_dinucl-shuffled_CV-5_20251030_23:...,/data/home/natant/Negatives/Runs/Review_rerun/...,GM12878,USF2,dinucl-shuffled,5,K562
...,...,...,...,...,...,...,...
1735,GM12878_ATF3_neighbors_CV-1_20251031_09:31_epo...,/data/home/natant/Negatives/Runs/Review_rerun/...,GM12878,ATF3,neighbors,1,A549
1736,GM12878_ATF3_shuffled_CV-0_20251031_09:15_epoc...,/data/home/natant/Negatives/Runs/Review_rerun/...,GM12878,ATF3,shuffled,0,A549
1737,GM12878_ATF3_shuffled_CV-2_20251031_09:19_epoc...,/data/home/natant/Negatives/Runs/Review_rerun/...,GM12878,ATF3,shuffled,2,A549
1738,GM12878_ATF3_neighbors_CV-3_20251031_09:38_epo...,/data/home/natant/Negatives/Runs/Review_rerun/...,GM12878,ATF3,neighbors,3,A549


In [ ]:
#all_runs_df.to_pickle('/data/home/natant/Negatives/testing_ground/20251118_all_cross_cell_runs.pkl')

In [9]:
all_runs_HQ = []
for cell_line in GM12878_overlaps:
    for tf in GM12878_overlaps[cell_line]:
        selected_runs = df_ckpt[(df_ckpt['cellline'] == 'GM12878') & (df_ckpt['TF'] == tf) & (df_ckpt['negative_type'] == 'HQ')]
        selected_runs["cellline_test"] = cell_line
        all_runs_HQ.append(selected_runs)
all_runs_HQ_df = pd.concat(all_runs_HQ, ignore_index=True)
all_runs_HQ_df

/tmp/ipykernel_1168808/3213720869.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  selected_runs["cellline_test"] = cell_line
/tmp/ipykernel_1168808/3213720869.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  selected_runs["cellline_test"] = cell_line
/tmp/ipykernel_1168808/3213720869.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/p

,filename,file_path,cellline,TF,negative_type,cv_split,cellline_test
0,GM12878_USF2_CV-5_20251103_15:58_epoch=29_val_...,/data/home/natant/Negatives/Runs/Review_rerun/...,GM12878,USF2,HQ,5,K562
1,GM12878_USF2_CV-4_20251103_15:58_epoch=29_val_...,/data/home/natant/Negatives/Runs/Review_rerun/...,GM12878,USF2,HQ,4,K562
2,GM12878_USF2_CV-0_20251103_15:56_epoch=34_val_...,/data/home/natant/Negatives/Runs/Review_rerun/...,GM12878,USF2,HQ,0,K562
3,GM12878_USF2_CV-2_20251103_15:57_epoch=13_val_...,/data/home/natant/Negatives/Runs/Review_rerun/...,GM12878,USF2,HQ,2,K562
4,GM12878_USF2_CV-1_20251103_15:57_epoch=21_val_...,/data/home/natant/Negatives/Runs/Review_rerun/...,GM12878,USF2,HQ,1,K562
...,...,...,...,...,...,...,...
343,GM12878_ATF3_CV-1_20251103_16:41_epoch=58_val_...,/data/home/natant/Negatives/Runs/Review_rerun/...,GM12878,ATF3,HQ,1,A549
344,GM12878_ATF3_CV-4_20251103_16:42_epoch=45_val_...,/data/home/natant/Negatives/Runs/Review_rerun/...,GM12878,ATF3,HQ,4,A549
345,GM12878_ATF3_CV-0_20251103_16:40_epoch=41_val_...,/data/home/natant/Negatives/Runs/Review_rerun/...,GM12878,ATF3,HQ,0,A549
346,GM12878_ATF3_CV-5_20251103_16:43_epoch=30_val_...,/data/home/natant/Negatives/Runs/Review_rerun/...,GM12878,ATF3,HQ,5,A549


In [ ]:
#all_runs_HQ_df.to_pickle('/data/home/natant/Negatives/testing_ground/REVIEW_cross_cell_perf/20251120_all_cross_cell_runs_HQ.pkl')

In [15]:
all_runs_df['file_path'].unique().shape[0]

900

In [16]:
all_runs_HQ_df['file_path'].unique().shape[0]

180

In [11]:
len(ckpt_files)

1428

In [20]:
# total checkpoint files for GM12878
print(df_ckpt["cellline"].value_counts())
# total unique checkpoints used for cross-cell evaluation
print(all_runs_df['file_path'].unique().shape[0] + all_runs_HQ_df['file_path'].unique().shape[0])

cellline
GM12878    1296
HepG2       132
Name: count, dtype: int64
1080


So these missing checkpoint just don't have any overlapping TFs with the other cell lines?

In [27]:
all_runs_df = pd.read_pickle('/data/home/natant/Negatives/testing_ground/20251118_all_cross_cell_runs.pkl')
all_runs_df

,filename,file_path,cellline,TF,negative_type,cv_split,cellline_test
0,GM12878_JunD_dinucl-sampled_CV-5_20251031_11:1...,/data/home/natant/Negatives/Runs/Review_rerun/...,GM12878,JunD,dinucl-sampled,5,K562
1,GM12878_JunD_shuffled_CV-3_20251031_11:42_epoc...,/data/home/natant/Negatives/Runs/Review_rerun/...,GM12878,JunD,shuffled,3,K562
2,GM12878_JunD_neighbors_CV-3_20251031_11:52_epo...,/data/home/natant/Negatives/Runs/Review_rerun/...,GM12878,JunD,neighbors,3,K562
3,GM12878_JunD_celltype_CV-3_20251103_22:52_epoc...,/data/home/natant/Negatives/Runs/Review_rerun/...,GM12878,JunD,celltype,3,K562
4,GM12878_JunD_dinucl-sampled_CV-1_20251031_11:1...,/data/home/natant/Negatives/Runs/Review_rerun/...,GM12878,JunD,dinucl-sampled,1,K562
...,...,...,...,...,...,...,...
1735,GM12878_SIX5_shuffled_CV-1_20251030_19:49_epoc...,/data/home/natant/Negatives/Runs/Review_rerun/...,GM12878,SIX5,shuffled,1,A549
1736,GM12878_SIX5_shuffled_CV-4_20251030_19:52_epoc...,/data/home/natant/Negatives/Runs/Review_rerun/...,GM12878,SIX5,shuffled,4,A549
1737,GM12878_SIX5_shuffled_CV-0_20251030_19:48_epoc...,/data/home/natant/Negatives/Runs/Review_rerun/...,GM12878,SIX5,shuffled,0,A549
1738,GM12878_SIX5_shuffled_CV-5_20251030_19:54_epoc...,/data/home/natant/Negatives/Runs/Review_rerun/...,GM12878,SIX5,shuffled,5,A549


In [28]:
run_tuples = [(row['file_path'], row['TF'], row['negative_type'], row['cv_split'], row['cellline_test']) 
              for _, row in all_runs_df.iterrows()]

In [ ]:
queue = Queue()

# Populate the queue with cell_tf_neg_combinations
for combination in run_tuples:
    queue.put(combination)

# Function to process combinations from the queue
def worker():
    while not queue.empty():
        file_path,tf, neg_mode, cv, cell_type = queue.get()
        command = [
            "python", 
            "/data/home/natant/Negatives/testing_ground/20251118_test_ckpt.py",
            "--ckpt_path", file_path,
            "--datafolder", datafolder,
            "--TF", tf, 
            "--celltype", cell_type, 
            "--neg_mode", neg_mode, 
            "--devices", "1",
            "--cross_val_set", str(cv),
            "--batch_size", "256",
            "--group_name", group_name
        ]
        subprocess.run(command)
        queue.task_done()

# Create and start threads
threads = []
for _ in range(max_concurrent_models):
    t = Thread(target=worker)
    t.start()
    threads.append(t)

# Wait for all threads to finish
for t in threads:
    t.join()